# BlindSpotter GNN 학습
## graph_dataset.pkl → GraphSAGE 학습

**사전 준비:** `graph_dataset.pkl`을 Drive의 `그기마(team 6)/BlindSpotter/` 폴더에 업로드

**데이터셋 스펙**
- 총 37,940 샘플 (train 28,190 / val 1,928 / test 7,822)
- Positive rate ~41% (킥보드/자전거가 사각지대에서 출현)
- 노드 피처: 8차원 `[x, y, vx, vy, heading, type_id, speed, is_occluder]`
- 엣지 피처: 5차원 `[distance, rel_vx, rel_vy, rel_heading, visibility_blocked]`

In [ ]:
# ── 0-A. 패키지 설치 ──────────────────────────────────────────
!pip install -q torch torch-geometric scikit-learn matplotlib
print('설치 완료')

In [ ]:
# ── 0-B. Google Drive 마운트 ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료')

In [ ]:
# ── 1. 데이터 로드 ────────────────────────────────────────────
import pickle
from pathlib import Path

# ▼▼▼ Drive 경로 확인 후 수정
DATA_PATH = '/content/drive/MyDrive/그기마(team 6)/BlindSpotter/graph_dataset.pkl'

with open(DATA_PATH, 'rb') as f:
    bundle = pickle.load(f)

dataset   = bundle['dataset']           # {'train': [...], 'val': [...], 'test': [...]}
NODE_FEAT = bundle['node_feature_names']
EDGE_FEAT = bundle['edge_feature_names']

print(f'노드 피처: {NODE_FEAT}')
print(f'엣지 피처: {EDGE_FEAT}')
for sp, gs in dataset.items():
    n_pos = sum(g['label'] for g in gs)
    print(f'  {sp:5s}: {len(gs):,}개  positive={n_pos:,} ({100*n_pos/max(len(gs),1):.1f}%)')

In [ ]:
# ── 2. PyG Data 변환 ──────────────────────────────────────────
import torch
from torch_geometric.data import Data, DataLoader

def to_pyg(g: dict) -> Data:
    x          = torch.tensor(g['x'],          dtype=torch.float)
    edge_index = torch.tensor(g['edge_index'], dtype=torch.long)
    edge_attr  = torch.tensor(g['edge_attr'],  dtype=torch.float) if g['edge_attr'] else torch.zeros(0, 5)
    y          = torch.tensor([g['label']],    dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

train_data = [to_pyg(g) for g in dataset['train']]
val_data   = [to_pyg(g) for g in dataset['val']]
test_data  = [to_pyg(g) for g in dataset['test']]

BATCH = 64
train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH)
test_loader  = DataLoader(test_data,  batch_size=BATCH)

print(f'변환 완료  train={len(train_data):,}  val={len(val_data):,}  test={len(test_data):,}')
print(f'샘플 노드 피처 shape: {train_data[0].x.shape}')
print(f'샘플 엣지 shape:      {train_data[0].edge_index.shape}')

In [ ]:
# ── 3. GraphSAGE 모델 정의 ────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool

class BlindZoneGNN(nn.Module):
    def __init__(self, node_dim=8, hidden=64, n_layers=3, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        in_dim = node_dim
        for _ in range(n_layers):
            self.convs.append(SAGEConv(in_dim, hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
            in_dim = hidden
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, ei)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BlindZoneGNN(node_dim=8, hidden=64, n_layers=3).to(DEVICE)
print(f'디바이스: {DEVICE}')
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 4. 학습 ───────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, f1_score

EPOCHS = 30
LR     = 1e-3

# Positive rate ~41% → class weight 적용
pos_weight = torch.tensor([1.43]).to(DEVICE)   # (1 - 0.41) / 0.41
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_y, all_p = 0.0, [], []
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            loss   = criterion(logits, batch.y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            all_y.extend(batch.y.cpu().tolist())
            all_p.extend(torch.sigmoid(logits).cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    auc  = roc_auc_score(all_y, all_p)
    pred = [1 if p > 0.5 else 0 for p in all_p]
    f1   = f1_score(all_y, pred)
    return avg_loss, auc, f1

best_val_auc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_auc, tr_f1 = run_epoch(train_loader, train=True)
    vl_loss, vl_auc, vl_f1 = run_epoch(val_loader,   train=False)
    scheduler.step()
    history.append((tr_loss, tr_auc, vl_loss, vl_auc))

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        torch.save(model.state_dict(), '/tmp/best_model.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train loss={tr_loss:.4f} auc={tr_auc:.4f} f1={tr_f1:.4f}  |  '
              f'val loss={vl_loss:.4f} auc={vl_auc:.4f} f1={vl_f1:.4f}')

print(f'\n최고 val AUC: {best_val_auc:.4f}')

In [ ]:
# ── 5. 테스트 평가 ────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('/tmp/best_model.pt', map_location=DEVICE))
te_loss, te_auc, te_f1 = run_epoch(test_loader, train=False)

print(f'Test  loss={te_loss:.4f}  AUC={te_auc:.4f}  F1={te_f1:.4f}')

# 상세 리포트
model.eval()
all_y, all_p = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        probs = torch.sigmoid(model(batch)).cpu().tolist()
        all_y.extend(batch.y.cpu().tolist())
        all_p.extend(probs)

pred = [1 if p > 0.5 else 0 for p in all_p]
print('\n' + classification_report(all_y, pred, target_names=['은폐', '출현']))

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([h[0] for h in history], label='train')
axes[0].plot([h[2] for h in history], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot([h[1] for h in history], label='train')
axes[1].plot([h[3] for h in history], label='val')
axes[1].set_title('AUC'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. 모델 Drive 저장 ────────────────────────────────────────
import shutil

SAVE_DIR = '/content/drive/MyDrive/그기마(team 6)/BlindSpotter/'
shutil.copy('/tmp/best_model.pt', SAVE_DIR + 'best_model.pt')
print(f'모델 저장 완료 → {SAVE_DIR}best_model.pt')